In [2]:
import pygrib
import bz2
import numpy.ma as ma
import os
import csv

compressed_path = "../data/downloaded_grib_files_regular/icon-d2_germany_regular-lat-lon_single-level_2023082015_000_2d_t_2m.grib2.bz2"

# Decompress the file and read the decompressed data
decompressed_path = compressed_path[:-4]  # Remove the '.bz2' extension
with bz2.BZ2File(compressed_path, 'rb') as compressed_file, open(decompressed_path, 'wb') as decompressed_file:
    decompressed_file.write(compressed_file.read())

# Open the decompressed GRIB file
gribfile = pygrib.open(decompressed_path)

# Select the desired variable (2 metre temperature in this case)
selected_grb = gribfile.select(name='2 metre temperature')[0]

# Get the data, latitudes, and longitudes
data_actual, lats, lons = selected_grb.data()

# Convert temperature values from Kelvin to Celsius
data_actual_celsius = data_actual - 273.15  # Conversion from Kelvin to Celsius

# Close the GRIB file
gribfile.close()

# Create a CSV file to write the data
csv_file_path = "output_data.csv"
with open(csv_file_path, 'w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    
    # Write the header row
    csv_writer.writerow(['Latitude', 'Longitude', 'Temperature (Celsius)'])

    # Loop through the data and write each row
    for i in range(len(lats)):
        for j in range(len(lons)):
            if not ma.getmask(data_actual[i, j]):
                csv_writer.writerow([lats[i, j], lons[i, j], data_actual_celsius[i, j]])

# Remove the decompressed GRIB file
os.remove(decompressed_path)

print("CSV file saved:", csv_file_path)


CSV file saved: output_data.csv
